In [1]:
import requests
from collections import defaultdict

RPC_CHAIN_URL = "http://192.168.4.80:26657"
REST_CHAIN_URL = "http://192.168.4.80:1317"


def get_current_epoch_data(height=None):
    headers = {}
    if height is not None:
        headers["X-Cosmos-Block-Height"] = str(height)

    res = requests.get(f"{REST_CHAIN_URL}/productscience/inference/inference/current_epoch_group_data", headers=headers)
    return res.json()


def get_group(group_id, height=None):
    headers = {}
    if height is not None:
        headers["X-Cosmos-Block-Height"] = str(height)

    res = requests.get(f"{REST_CHAIN_URL}/cosmos/group/v1/group_members/{group_id}?pagination.limit=10000", headers=headers)
    return res.json()

# PoC 155


## Getting generated batches

In [2]:
res = requests.get(f"{REST_CHAIN_URL}/productscience/inference/inference/poc_batches_for_stage/2396960").json()

In [3]:
participant_to_batches = defaultdict(list)


for participant in res["poc_batch"]:
    batch = participant["poc_batch"]
    participant_to_batches[participant["participant"]].append(batch)

def get_stats(participant):
    batches = participant_to_batches[participant]
    all_nonces = [nonce for batch_list in batches for batch in batch_list for nonce in batch["nonces"]]
    all_distances = [dist for batch_list in batches for batch in batch_list for dist in batch["dist"]]
    mlnode_to_batches = defaultdict(list)
    for batch_list in batches:
        for batch in batch_list:
            mlnode_to_batches[batch["node_id"]].extend(batch["nonces"])

    return all_nonces, all_distances, mlnode_to_batches


participant_to_stats = defaultdict(dict)
for participant in res["poc_batch"]:
    all_nonces, all_distances, mlnode_to_batches = get_stats(participant["participant"])
    participant_to_stats[participant["participant"]] = {
        "all_nonces": all_nonces,
        "all_distances": all_distances,
        "mlnode_to_batches": mlnode_to_batches
    }

## Getting validations

In [4]:
val_res = requests.get(f"{REST_CHAIN_URL}/productscience/inference/inference/poc_validations_for_stage/2396960").json()

participant_to_validation = defaultdict(lambda: defaultdict(list))
for validation in val_res["poc_validation"]:
    participant = validation["participant"]
    for validation in validation["poc_validation"]:
        to_validate = validation["participant_address"]
        assert to_validate == participant
        validator = validation["validator_participant_address"]
        fraud_detected = bool(validation["fraud_detected"])

        if fraud_detected:
            participant_to_validation[participant]["invalid"].append(validator)
        else:
            participant_to_validation[participant]["valid"].append(validator)    

In [5]:
current_epoch_data = get_current_epoch_data(2396960)
participant_to_weight = {x["member_address"]: x["weight"] for x in current_epoch_data["epoch_group_data"]["validation_weights"]}

group = get_group(788, 2396960)
participant_to_weight = {x["member"]["address"]: x["member"]["weight"] for x in group["members"]}

In [6]:
total_weight = sum(int(x) for x in participant_to_weight.values())
total_weight

7261934

## Total weight of validators per each participant 

In [7]:
counter = 0
for participant, validators in sorted(participant_to_validation.items(), key=lambda x: len(participant_to_stats[x[0]]["all_nonces"]), reverse=True):
    total_valid = sum(int(participant_to_weight.get(validator, 0)) for validator in validators["valid"])
    total_invalid = sum(int(participant_to_weight.get(validator, 0)) for validator in validators["invalid"])
    print(counter, participant, total_valid, total_invalid)
    counter += 1


0 gonka1w6kt2aj02du25kuc9wsza94h9l7exa0uf64fjq 3571493 0
1 gonka1r34p353cxrvxf3x29raz0x8axflen82a04env4 3560322 5824
2 gonka14a0856a3q78pmesxpc946xm9nsj3f275l9f5pa 3660585 0
3 gonka1l44nacmpmt83kavvevmhlh8elspjtjn6wvwzm0 3569096 0
4 gonka1nkmwp905ka4dzvuwvdaudgjj7yjk8h5aslfw73 3531651 0
5 gonka1d0cekvf368psxff6ur7pl42vvs225rzu9a76nj 3595050 0
6 gonka1lecmns7dj5wm8f53pe6c8nueukqafknel2wt2m 3539854 0
7 gonka1qmpm8tsynnfkqe9902dd6ny4lc5hvp8q7wkrpc 3528769 0
8 gonka1w6wwv3wq25p8qge4lqsnfzs8lsd3s8ty6au65p 4467911 0
9 gonka1s5vtmnn8tqvqfh7gd9hv48kkt2t4mkh7s85zh6 4564743 0
10 gonka10etnufq85u67k075yuxq6h3rzwlcln5rffhlyx 4347492 0
11 gonka1lrls7q8nu8rhjgchctswfsjlnjh84vwycw3jgt 4488153 0
12 gonka1sg2uj8u59hjkzh6qwfsgg0n0w63ux35my80h4a 3696675 0
13 gonka178p3vw9zxs885l4a29g3sep0v9uplfq4pzvmrq 4458959 0
14 gonka138qmc42pq75mxf3r3fh9k35ukvj9frmvuc7f0p 3669984 0
15 gonka135yst4re34z3rlfquhknqxvnaj35pjwu0mw622 3670504 0
16 gonka100k8hr43z5vp0fnyc94m7lum6mj4st6vksxz8s 3765024 0
17 gonka155cnj622zfdl

In [8]:
all_participant_with_weight = [participant for participant in participant_to_stats if int(participant_to_weight.get(participant, 0)) > 0]

print(
    f"Participant who had weight in epoch 154 and participated in PoC for epoch 155:\n"
    f" length: {len(all_participant_with_weight)}\n"
    f" weight: {sum(int(participant_to_weight.get(validator, 0)) for validator in all_participant_with_weight)}\n"
)

Participant who had weight in epoch 154 and participated in PoC for epoch 155:
 length: 185
 weight: 6328708



In [9]:
all_ever_voted = set()

for validators in participant_to_validation.values():
    all_ever_voted.update(validators["valid"])
    all_ever_voted.update(validators["invalid"])


all_ever_voted_with_weight = [participant for participant in all_ever_voted if int(participant_to_weight.get(participant, 0)) > 0]

print(
    f"Participant who had weight in epoch 154 and validated PoC for epoch 155:\n"
    f" length: {len(all_ever_voted_with_weight)}\n"
    f" weight: {sum(int(participant_to_weight.get(validator, 0)) for validator in all_ever_voted_with_weight)}\n"
)

Participant who had weight in epoch 154 and validated PoC for epoch 155:
 length: 171
 weight: 4663318



In [10]:
not_voted = []
for previous_participant in all_participant_with_weight:
    if previous_participant not in participant_to_weight:
        continue

    if previous_participant not in all_ever_voted_with_weight:
        not_voted.append(previous_participant)
        continue

print(
    len(not_voted),
    sum(int(participant_to_weight.get(validator, 0)) for validator in not_voted)
)

print(
    f"Participant who had weight in epoch 154 and NOT validated PoC for epoch 155:\n"
    f" length: {len(not_voted)}\n"
    f" weight: {sum(int(participant_to_weight.get(validator, 0)) for validator in not_voted)}\n"
)

for idx, participant in enumerate(not_voted):
    print(f"{idx:02d}: {participant}")


15 1814973
Participant who had weight in epoch 154 and NOT validated PoC for epoch 155:
 length: 15
 weight: 1814973

00: gonka1d0cekvf368psxff6ur7pl42vvs225rzu9a76nj
01: gonka1qmpm8tsynnfkqe9902dd6ny4lc5hvp8q7wkrpc
02: gonka1fxdt48vp78uxa7apuuamv4clwafxagnjg9eulc
03: gonka1lmgutdlel0fny79zvlsh9aw3c0dw75gtscka3g
04: gonka1000rv0ddp9yk6djr4y29mt0gu3m4p8mg4kmjcv
05: gonka1000xmydnfvphwy4n5yww4ex6nwk9mqslf2gnhs
06: gonka199lgrq8l9xcqqnr0agajzl78c4dpfvwnsc4elm
07: gonka1nkmwp905ka4dzvuwvdaudgjj7yjk8h5aslfw73
08: gonka1lecmns7dj5wm8f53pe6c8nueukqafknel2wt2m
09: gonka1zv89w02sdfzwy74s9vcthk38dnfsw36s4h7s93
10: gonka1v5ggga7lslfg2e57m9anxud40v2s4t9dw8yj68
11: gonka1000xjetteyu7gy726vnhdxq69y3meq04gvnk4d
12: gonka14a0856a3q78pmesxpc946xm9nsj3f275l9f5pa
13: gonka1hkz4qdpl2z4zvlyt6mxfsx04yxfp7sjhnyu2xn
14: gonka1l44nacmpmt83kavvevmhlh8elspjtjn6wvwzm0


In [12]:
# Build reverse mapping: validator -> set of participants they validated
validator_to_validated_participants = defaultdict(set)

for participant, validators in participant_to_validation.items():
    for validator in validators["valid"]:
        validator_to_validated_participants[validator].add(participant)
    for validator in validators["invalid"]:
        validator_to_validated_participants[validator].add(participant)

# Get epoch 155 weights (current epoch data)
epoch_155_data = get_current_epoch_data()
participant_to_weight_155 = {x["member_address"]: x["weight"] for x in epoch_155_data["epoch_group_data"]["validation_weights"]}

# Filter to validators who had weight in epoch 154 and count validations
validator_validation_counts = {}
for validator in participant_to_weight:
    if int(participant_to_weight.get(validator, 0)) > 0:
        count = len(validator_to_validated_participants.get(validator, set()))
        validator_validation_counts[validator] = count

# Sort by count (descending to show top validators first)
sorted_validators = sorted(validator_validation_counts.items(), key=lambda x: x[1], reverse=True)

print("Participants who had weight in epoch 154 and how many participants they validated in epoch 155:\n")
print(f"{'address':<45} {'validated':>10} {'weight_154':>12} {'generated_nonces':>14} {'weight_155':>12}")
print("-" * 95)
for validator, count in sorted_validators:
    weight_154 = participant_to_weight.get(validator, 0)
    nonces_claimed = len(participant_to_stats.get(validator, {}).get("all_nonces", []))
    weight_155 = participant_to_weight_155.get(validator, 0)
    is_suspicious = nonces_claimed > 0 and int(weight_154) > 0 and count < 197

    # Check work done by host at PoC validation
    validated_nonces = 200*count
    # 1 valid nonce in 50 generated
    checked_raw_nonces_at_validation = int(nonces_claimed) * 50
    # validation is twice longer
    validation_capability = checked_raw_nonces_at_validation * 2 
    # let's asssume validation is 100 times less effective then generation (which is not true)
    validation_capability = validation_capability / 100.

    # Check if the host did not do enough work
    is_suspicious = is_suspicious and (validated_nonces < validation_capability)
    print(f"{validator:<45} {count:>10} {weight_154:>12} {nonces_claimed:>14} {weight_155:>12} {'SUSPICIOUS' if is_suspicious else ''}")

Participants who had weight in epoch 154 and how many participants they validated in epoch 155:

address                                        validated   weight_154 generated_nonces   weight_155
-----------------------------------------------------------------------------------------------
gonka1q6v5taltnxxzjrgpheu5t7pfjwmsu6x8gs24sw         197         3288           1401         3491 
gonka1py4j23jhz2nah9d8lqpxn2lq6e07lx6e6jmaym         197         3572           1609            0 
gonka1pf4z32nqz5ax7zsxsmf4tan8wsnp957y9t4qtt         197         3176            999         2488 
gonka1p2959dx973hd57qsalxvesrcv649296x90ry76         197        90775          40289       100508 
gonka1p209mlvngz8hu0p5ff7dsjzqhjpx4jzn6k652r         197        49215          33549            0 
gonka1zpm8qujddac84y7wfh7h3p85ys336n9zjqckzl         197         3610           1706            0 
gonka1zrnrd7zcqnhjytqa8zsg63slxt2g45ctlqy3fm         197         3210           1070            0 
gonka1zsnpv9ht